# PSyle Checker- font

## Load and Explore Dataset

In [1]:
import pandas as pd

# Specify the path to your CSV file
file_path = "../data/image_pairs_labels.csv"

# Read the CSV file into a DataFrame
df = pd.read_csv(file_path)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)


# Display the first few rows of the DataFrame
print(df.head())

# Assuming the DataFrame `df` has the same structure
image_pairs = list(zip(df['img1_path'], df['img2_path']))
style_labels = df['style_label'].tolist()
font_labels = df['font_label'].tolist()

# Now `image_pairs`, `style_labels`, and `font_labels` are reconstructed

df['original_index'] = df.index

                                        img1_path  \
0  C:\Users\maria\Downloads\dataset\img1_3163.png   
1  C:\Users\maria\Downloads\dataset\img1_1868.png   
2   C:\Users\maria\Downloads\dataset\img1_102.png   
3   C:\Users\maria\Downloads\dataset\img1_873.png   
4  C:\Users\maria\Downloads\dataset\img1_2750.png   

                                        img2_path  style_label  font_label  
0  C:\Users\maria\Downloads\dataset\img2_3163.png            0           1  
1  C:\Users\maria\Downloads\dataset\img2_1868.png            1           1  
2   C:\Users\maria\Downloads\dataset\img2_102.png            0           0  
3   C:\Users\maria\Downloads\dataset\img2_873.png            0           1  
4  C:\Users\maria\Downloads\dataset\img2_2750.png            0           0  


In [2]:
import numpy as np
from PIL import Image # Import the PIL library


# Example training data (this is just a placeholder)
# image_pairs = [(img1, img2), (img3, img4), ...]
# style_labels = [0, 1, ...]  # Binary labels for style coherence
# font_labels = [0, 1, ...]   # Binary labels for font similarity

# Convert to numpy arrays
image_pairs = np.array(image_pairs)
style_labels = np.array(style_labels)
font_labels = np.array(font_labels)

# Function to load and preprocess an image
def load_and_preprocess_image(image_path):
    img = Image.open(image_path)
    img = img.resize((224, 224))  # Resize to match input shape
    img = np.array(img) / 255.0  # Normalize pixel values
    return img

# Load and preprocess the images
images_1 = np.array([load_and_preprocess_image(path) for path in image_pairs[:, 0]])
images_2 = np.array([load_and_preprocess_image(path) for path in image_pairs[:, 1]])


In [3]:
len(font_labels)

6000

## Model training single lable output

In [4]:
import time
import gc
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.layers import (BatchNormalization, Dropout, GlobalAveragePooling2D, 
                                     Dense, Conv2D, MaxPooling2D, Input, Concatenate)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers
from tensorflow.keras import backend as K

# Create a CNN branch for each input
def create_cnn_branch(input_shape, branch_name, input_tensor):
    x = Conv2D(32, (3, 3), activation='relu', padding='same', name=f'{branch_name}_conv1')(input_tensor)
    x = BatchNormalization()(x)
    x = MaxPooling2D(pool_size=(2, 2), name=f'{branch_name}_pool1')(x)

    x = Conv2D(64, (3, 3), activation='relu', padding='same', name=f'{branch_name}_conv2')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(pool_size=(2, 2), name=f'{branch_name}_pool2')(x)

    x = Conv2D(128, (3, 3), activation='relu', padding='same', name=f'{branch_name}_conv3')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(pool_size=(2, 2), name=f'{branch_name}_pool3')(x)

    x = Conv2D(256, (3, 3), activation='relu', padding='same', name=f'{branch_name}_conv4')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(pool_size=(2, 2), name=f'{branch_name}_pool4')(x)

    x = GlobalAveragePooling2D(name=f'{branch_name}_global_pool')(x)
    x = Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'{branch_name}_fc')(x)
    x = Dropout(0.5)(x)

    return x

# Build the full dual-branch model
def create_dual_branch_model(input_shape):
    input_1 = Input(shape=input_shape, name='input_1')
    input_2 = Input(shape=input_shape, name='input_2')

    branch_output_1 = create_cnn_branch(input_shape, "branch1", input_1)
    branch_output_2 = create_cnn_branch(input_shape, "branch2", input_2)

    combined = Concatenate(name='concatenate')([branch_output_1, branch_output_2])

    x = Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.01), name='fc_combined_1')(combined)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)

    x = Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.01), name='fc_combined_2')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)

    x = Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.01), name='fc_combined_3')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)

    style_output = Dense(1, activation='sigmoid', name='style_output')(x)

    model = Model(inputs=[input_1, input_2], outputs=style_output)

    model.compile(
        optimizer=Adam(learning_rate=3e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

# Callbacks
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Split data into train/test
X_train_1, X_test_1, X_train_2, X_test_2, y_train, y_test, train_indices, test_indices = train_test_split(
    images_1, images_2, font_labels, df['original_index'].values, test_size=0.2, random_state=42, stratify=font_labels
)

# K-Fold cross-validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train_1, y_train)):
    print(f"\nTraining on fold {fold + 1}")

    # Free memory and clear session
    K.clear_session()
    gc.collect()

    # Build a new model
    model = create_dual_branch_model(input_shape=(224, 224, 3))

    # Prepare data
    X_train_fold_1 = np.array(X_train_1)[train_idx].astype(np.float32)
    X_train_fold_2 = np.array(X_train_2)[train_idx].astype(np.float32)
    X_val_fold_1 = np.array(X_train_1)[val_idx].astype(np.float32)
    X_val_fold_2 = np.array(X_train_2)[val_idx].astype(np.float32)
    y_train_fold = np.array(y_train)[train_idx]
    y_val_fold = np.array(y_train)[val_idx]

    # Fit the model
    history = model.fit(
        [X_train_fold_1, X_train_fold_2],
        y_train_fold,
        validation_data=([X_val_fold_1, X_val_fold_2], y_val_fold),
        epochs=100,
        batch_size=16,
        callbacks=[reduce_lr, early_stopping],
        verbose=1
    )

    print(f"Fold {fold + 1} completed.")

end_time = time.time()
print("Total training time: ", round(end_time - start_time, 2), "seconds")

# Final evaluation on test set
X_test_1 = np.array(X_test_1).astype(np.float32)
X_test_2 = np.array(X_test_2).astype(np.float32)

test_loss, test_accuracy = model.evaluate([X_test_1, X_test_2], y_test, verbose=0)
print(f"\nTest accuracy: {test_accuracy:.4f}")



Training on fold 1
Epoch 1/100
240/240 [==============================] - 21s 66ms/step - loss: 15.1292 - accuracy: 0.5560 - val_loss: 13.7860 - val_accuracy: 0.7760 - lr: 3.0000e-04
Epoch 2/100
240/240 [==============================] - 14s 60ms/step - loss: 13.0900 - accuracy: 0.6310 - val_loss: 11.9816 - val_accuracy: 0.7760 - lr: 3.0000e-04
Epoch 3/100
240/240 [==============================] - 14s 60ms/step - loss: 11.3013 - accuracy: 0.6724 - val_loss: 10.3554 - val_accuracy: 0.7760 - lr: 3.0000e-04
Epoch 4/100
240/240 [==============================] - 14s 60ms/step - loss: 9.7276 - accuracy: 0.6893 - val_loss: 8.9092 - val_accuracy: 0.7760 - lr: 3.0000e-04
Epoch 5/100
240/240 [==============================] - 14s 60ms/step - loss: 8.3111 - accuracy: 0.7156 - val_loss: 7.5794 - val_accuracy: 0.7781 - lr: 3.0000e-04
Epoch 6/100
240/240 [==============================] - 14s 60ms/step - loss: 7.0688 - accuracy: 0.7232 - val_loss: 6.4075 - val_accuracy: 0.7760 - lr: 3.0000e-04
Ep

In [5]:
import os

def save_all_models(model, model_name,category, save_dir="saved_models"):
    os.makedirs(save_dir, exist_ok=True)

    model_path = os.path.join(save_dir, model_name)
    model.save(model_path)
    print(f"Saved model for category '{category}' at '{model_path}'")
        
save_dir="../exported_model"
model_name = "pstylechecker_fontmodel"
category= "font_coherence"
save_all_models(model,model_name, category, save_dir)


INFO:tensorflow:Assets written to: ../exported_model\pstylechecker_fontmodel\assets


INFO:tensorflow:Assets written to: ../exported_model\pstylechecker_fontmodel\assets


Saved model for category 'font_coherence' at '../exported_model\pstylechecker_fontmodel'


In [ ]:
## Evaluationm

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, jaccard_score, cohen_kappa_score, roc_auc_score, roc_curve, confusion_matrix
import json
# Predict on the validation set
style_pred_val = model.predict([X_test_1, X_test_2])

# Convert predictions to binary values (threshold at 0.5 for binary classification)
style_pred_val_binary = (style_pred_val > 0.5).astype(int)

# Identify False Positives
false_positive_indices = [
    test_indices[i] for i, (true, pred) in enumerate(zip(y_test, style_pred_val_binary)) if true == 0 and pred == 1
]

# Analyze the corresponding attributes
# for idx in false_positive_indices:
#     attributes_img1 = json.loads(df.loc[idx, 'attributes_img1'])
#     attributes_img2 = json.loads(df.loc[idx, 'attributes_img2'])
#     print(f"False Positive Pair {idx}:")
#     print(f"Attributes Image 1: {attributes_img1}")
#     print(f"Attributes Image 2: {attributes_img2}")


# Calculate additional metrics using sklearn
style_accuracy = accuracy_score(y_test, style_pred_val_binary)
style_precision = precision_score(y_test, style_pred_val_binary)
style_recall = recall_score(y_test, style_pred_val_binary)
style_f1 = f1_score(y_test, style_pred_val_binary)

print(f"Style Output - Accuracy: {style_accuracy:.2f}, Precision: {style_precision:.2f}, Recall: {style_recall:.2f}, F1 Score: {style_f1:.2f}")

# Compute Jaccard similarity
jaccard = jaccard_score(y_test, style_pred_val_binary)
print("Jaccard Similarity: ", jaccard)

# Calculate Cohen's Kappa
kappa = cohen_kappa_score(y_test, style_pred_val_binary)
print("Cohen's Kappa: ", kappa)

# Calculate ROC AUC
roc_auc = roc_auc_score(y_test, style_pred_val)
print("ROC AUC: ", roc_auc)

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, style_pred_val)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')  # Diagonal line for random classifier
plt.title('ROC Curve')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

# Compute and plot confusion matrix
cm = confusion_matrix(y_test, style_pred_val_binary)

# Plot confusion matrix
plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Predicted: 0", "Predicted: 1"], yticklabels=["Actual: 0", "Actual: 1"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

